## Funciones comunes (eclib)

In [1]:
import os, json, time, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2

CLASS_NAMES = ["G-cocci", "G+cocci", "G-bacilli", "G+bacilli"]
GB, GPB = 2, 3          # ids de G-bacilli y G+bacilli
NMS_IOU = 0.7

def prepare_ds(src_root, dst_root="/kaggle/temp/yolo_ds", with_bg=False):
    """Copia el dataset a un lugar escribible y escribe el yaml con rutas reales."""
    src_root, dst_root = Path(src_root), Path(dst_root)
    if not dst_root.exists(): shutil.copytree(src_root, dst_root)
    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
    tr = "[images/train, images/bg_train]" if with_bg else "images/train"
    y = dst_root / ("data_bg.yaml" if with_bg else "data_base.yaml")
    y.write_text(f"path: {dst_root}\ntrain: {tr}\nval: images/val\ntest: images/test\nnames:\n{names}\n")
    return dst_root, y

def read_gt(label_path):
    a = np.loadtxt(label_path, ndmin=2) if Path(label_path).stat().st_size else np.zeros((0, 5))
    c = a[:, 0].astype(int); xc, yc, w, h = a[:, 1:5].T
    return c, np.stack([xc-w/2, yc-h/2, xc+w/2, yc+h/2], 1)

def predict_dets(model, img_paths, imgsz, conf=0.01, iou=NMS_IOU, batch=32, device=None, max_det=1000):
    """Detecciones de baja confianza; el barrido de umbral se hace despues sobre este cache."""
    out = {}
    paths = [str(p) for p in img_paths]
    for i in range(0, len(paths), batch):
        for p, r in zip(paths[i:i+batch], model.predict(paths[i:i+batch], imgsz=imgsz, conf=conf, iou=iou, verbose=False, device=device, max_det=max_det)):
            h, w = r.orig_shape
            b = r.boxes
            out[Path(p).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxy.cpu().numpy() / np.array([w, h, w, h]))
    return out

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    i = iw * ih; ua = ((a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]))[:, None] + ((b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1]))[None] - i
    return i / np.maximum(ua, 1e-9)

def load_gts(ds_root, split):
    return {p.stem: read_gt(p) for p in sorted((Path(ds_root) / "labels" / split).glob("*.txt"))}

def counts(dets, gts, conf, classes):
    """Conteos por imagen (gt, pred) de las clases dadas."""
    rows = []
    for k, (gc, _) in gts.items():
        c, s, _b = dets[k]
        rows.append((int(np.isin(gc, classes).sum()), int((np.isin(c, classes) & (s >= conf)).sum())))
    return np.array(rows)

def count_metrics(dets, gts, conf, classes):
    a = counts(dets, gts, conf, classes); e = a[:, 1] - a[:, 0]
    return dict(mae=float(np.abs(e).mean()), bias=float(e.mean()), exact=float((e == 0).mean()),
                gt_total=int(a[:, 0].sum()), pred_total=int(a[:, 1].sum()))

def boot_ci(dets, gts, conf, classes, n=1000, seed=0):
    a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0]); rng = np.random.default_rng(seed)
    m = [e[rng.integers(0, len(e), len(e))].mean() for _ in range(n)]
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def boot_ci_groups(dets, gts, conf, classes, group_of, n=1000, seed=0, stat="mae"):
    """IC 95 % remuestreando GRUPOS de origen (group_of: stem -> grupo), no imagenes."""
    keys = list(gts); a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0])
    g = np.array([group_of[k] for k in keys]); ug, inv = np.unique(g, return_inverse=True)
    sums = np.bincount(inv, weights=e, minlength=len(ug)); cnt = np.bincount(inv, minlength=len(ug)); rng = np.random.default_rng(seed)
    m = []
    for _ in range(n):
        ix = rng.integers(0, len(ug), len(ug)); m.append(sums[ix].sum() / cnt[ix].sum())
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def match_stats(dets, gts, conf, cls=GB, thr=0.5):
    """TP/FP/FN de la clase cls (IoU>=thr, 1 a 1 greedy) y confusiones con la otra clase de bacilos."""
    tp = fp = fn = conf_to_gpb = 0
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf
        pc, pb = c[m], b[m]
        g = gc == cls; gb_ = gbx[g]; p = pb[pc == cls]
        iou = box_iou(p, gb_); used_p, used_g = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(gb_))), reverse=True):
            if v < thr: break
            if i in used_p or j in used_g: continue
            used_p.add(i); used_g.add(j)
        t = len(used_p); tp += t; fp += len(p) - t; fn += len(gb_) - t
        other = GPB if cls == GB else GB
        un = [j for j in range(len(gb_)) if j not in used_g]
        if un:
            q = box_iou(pb[pc == other], gb_[un]); conf_to_gpb += int((q >= thr).any(0).sum()) if q.size else 0
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=2*P*R/max(P+R, 1e-9), gt_clase_predicha_como_otra=conf_to_gpb)

def sweep(dets, gts, classes, confs=np.round(np.arange(0.05, 0.81, 0.05), 2)):
    rows = []
    for c in confs:
        r = dict(conf=float(c), **count_metrics(dets, gts, c, classes))
        r.update({"f1_Gb": match_stats(dets, gts, c)["f1"]}); rows.append(r)
    return pd.DataFrame(rows)

def bg_fp(dets_bg, conf, classes=(GB,)):
    n = np.array([int((np.isin(c, classes) & (s >= conf)).sum()) for c, s, _ in dets_bg.values()])
    return dict(n_imgs=len(n), fp_por_imagen=float(n.mean()), imgs_con_fp=float((n > 0).mean()))

def cpu_latency_ms(model, img, imgsz, n=100, warm=10):
    for _ in range(warm): model.predict(img, imgsz=imgsz, device="cpu", verbose=False)
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); model.predict(img, imgsz=imgsz, device="cpu", verbose=False); t.append((time.perf_counter()-t0)*1000)
    return float(np.median(t))

def ap_table(m):
    """mAP50 y mAP50-95 por clase desde un objeto DetMetrics de ultralytics."""
    idx = list(m.box.ap_class_index)
    return {CLASS_NAMES[c]: dict(ap50=float(m.box.ap50[i]), ap=float(m.box.ap[i])) for i, c in enumerate(idx)}


# Paso 7 y 8 (CPU): seleccion por val, umbrales congelados, TEST una vez por modelo, exportacion y paridad
Regla pre-registrada en ACCEPTANCE.md: elegido = mayor F1 G-bacilli en val al conf de minimo MAE de conteo.

## Entorno, datos y pesos

In [2]:
import os, json, pickle, shutil, subprocess, sys, time, hashlib
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
subprocess.run("pip install -q ultralytics", shell=True)
from ultralytics import YOLO
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)
SRC = next(Path("/kaggle/input").glob("**/yolo_ds")); DSR = Path("/kaggle/temp/yolo_ds"); shutil.copytree(SRC, DSR)
MODELS = {"E3": 42, "E3s43": 43, "E3s44": 44}
WTS = {m: next(Path("/kaggle/input").glob(f"**/{m}_best.pt")) for m in MODELS}; print(WTS)
names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
sp = pd.read_csv(DSR / "splits.csv", dtype={"dhash": str}); sp["stem"] = sp.name.str.replace(".jpg", "", regex=False); INFO = sp.set_index("stem")
GROUP_OF = INFO.group.to_dict(); PATTERN = INFO.pattern.to_dict()
IM = {s: sorted((DSR / "images" / s).glob("*.jpg")) for s in ("val", "test")}; GT = {s: load_gts(DSR, s) for s in ("val", "test")}
(DSR / "data_base.yaml").write_text(f"path: {DSR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names}\n")
CONFS = np.round(np.arange(0.05, 0.701, 0.05), 2)

def match_boxes(dets, gts, conf, cls=GB, thr=0.5):
    """Por imagen: tp, fp, fn de la clase cls (IoU>=thr, 1 a 1 greedy)."""
    rows = []
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf; pc, pb = c[m], b[m]
        g = np.where(gc == cls)[0]; p = np.where(pc == cls)[0]; iou = box_iou(pb[p], gbx[g]); up, ug = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(g))), reverse=True):
            if v < thr: break
            if i in up or j in ug: continue
            up.add(i); ug.add(j)
        rows.append((k, len(up), len(p) - len(up), len(g) - len(ug), len(g), len(p)))
    return pd.DataFrame(rows, columns=["stem", "tp", "fp", "fn", "n_gt", "n_pred"]).set_index("stem")

def confusion(dets, gts, conf, thr=0.5):
    M = np.zeros((5, 5), int)
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf; pc, pb = c[m], b[m]; iou = box_iou(gbx, pb); ug, up = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(gc)) for j in range(len(pc))), reverse=True):
            if v < thr: break
            if i in ug or j in up: continue
            ug.add(i); up.add(j); M[gc[i], pc[j]] += 1
        for i in range(len(gc)):
            if i not in ug: M[gc[i], 4] += 1
        for j in range(len(pc)):
            if j not in up: M[4, pc[j]] += 1
    return pd.DataFrame(M, index=CLASS_NAMES + ["fondo"], columns=CLASS_NAMES + ["fondo"])

def summarize(mb):
    tp, fp, fn = mb.tp.sum(), mb.fp.sum(), mb.fn.sum(); P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(precision=float(P), recall=float(R), f1=float(2 * P * R / max(P + R, 1e-9)), mae=float((mb.n_pred - mb.n_gt).abs().mean()), bias=float((mb.n_pred - mb.n_gt).mean()), n_img=len(mb), n_gt=int(mb.n_gt.sum()))

def boot_groups(mb, n=2000, seed=0):
    """IC 95 % remuestreando grupos de origen: precision, recall, F1, MAE."""
    g = mb.index.map(GROUP_OF).to_numpy(); ug, inv = np.unique(g, return_inverse=True)
    agg = pd.DataFrame({"tp": mb.tp.to_numpy(), "fp": mb.fp.to_numpy(), "fn": mb.fn.to_numpy(), "ae": (mb.n_pred - mb.n_gt).abs().to_numpy(), "one": 1}).groupby(inv).sum().to_numpy()
    rng = np.random.default_rng(seed); res = []
    for _ in range(n):
        t = agg[rng.integers(0, len(ug), len(ug))].sum(0); P = t[0] / max(t[0] + t[1], 1); R = t[0] / max(t[0] + t[2], 1); res.append((P, R, 2 * P * R / max(P + R, 1e-9), t[3] / t[4]))
    q = np.percentile(np.array(res), [2.5, 97.5], axis=0)
    return {k: [float(q[0, i]), float(q[1, i])] for i, k in enumerate(["precision", "recall", "f1", "mae"])}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 5.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{'E3': PosixPath('/kaggle/input/notebooks/fabryzziomezatorres/ecoli-e2e3/outputs/E3_best.pt'), 'E3s43': PosixPath('/kaggle/input/notebooks/fabryzziomezatorres/ecoli-e3-s43/outputs/E3s43_best.pt'), 'E3s44': PosixPath('/kaggle/input/notebooks/fabryzziomezatorres/ecoli-e3-s44/outputs/E3s44_best.pt')}


## Paso 7a. Por semilla: inferencia val, umbrales por minimo MAE (val), seleccion

In [3]:
VAL = {}
for m, sd in MODELS.items():
    mod = YOLO(str(WTS[m])); d = {iou: predict_dets(mod, IM["val"], 320, conf=0.01, iou=iou, device="cpu") for iou in (0.5, 0.6, 0.7)}
    rows = []
    for iou, dd in d.items():
        for nm, cl in (("estricta", [GB]), ("amplia", [GB, GPB])):
            for c in CONFS: rows.append(dict(iou=iou, definicion=nm, conf=float(c), mae=count_metrics(dd, GT["val"], c, cl)["mae"]))
    g = pd.DataFrame(rows); fz = {}
    for nm in ("estricta", "amplia"):
        r = g[g.definicion == nm].sort_values(["mae", "iou", "conf"]).iloc[0]; fz[nm] = dict(iou=float(r.iou), conf=float(r.conf), mae_val=float(r.mae))
    mb = match_boxes(d[fz["estricta"]["iou"]], GT["val"], fz["estricta"]["conf"]); fz["val_estricta"] = summarize(mb)
    VAL[m] = fz; g.to_csv(OUT / f"{m}_rejilla_val.csv", index=False); print(m, json.dumps(fz))
CHOSEN = max(VAL, key=lambda m: VAL[m]["val_estricta"]["f1"]); print("ELEGIDO por val:", CHOSEN)
(OUT / "seleccion_val.json").write_text(json.dumps(dict(chosen=CHOSEN, val=VAL), indent=1))

E3 {"estricta": {"iou": 0.5, "conf": 0.6, "mae_val": 0.29591836734693877}, "amplia": {"iou": 0.5, "conf": 0.5, "mae_val": 0.35034013605442177}, "val_estricta": {"precision": 0.8502358490566038, "recall": 0.7931793179317932, "f1": 0.8207171314741036, "mae": 0.29591836734693877, "bias": -0.0691609977324263, "n_img": 882, "n_gt": 909}}
E3s43 {"estricta": {"iou": 0.5, "conf": 0.6, "mae_val": 0.28798185941043086}, "amplia": {"iou": 0.5, "conf": 0.45, "mae_val": 0.3469387755102041}, "val_estricta": {"precision": 0.8464163822525598, "recall": 0.8184818481848185, "f1": 0.8322147651006712, "mae": 0.28798185941043086, "bias": -0.034013605442176874, "n_img": 882, "n_gt": 909}}
E3s44 {"estricta": {"iou": 0.5, "conf": 0.6, "mae_val": 0.30839002267573695}, "amplia": {"iou": 0.5, "conf": 0.5, "mae_val": 0.3435374149659864}, "val_estricta": {"precision": 0.8578371810449574, "recall": 0.7766776677667767, "f1": 0.815242494226328, "mae": 0.30839002267573695, "bias": -0.09750566893424037, "n_img": 882, "n

1305

## Paso 7b. TEST una sola vez por modelo congelado
Se corre ultralytics val (mAP50 por clase) y la inferencia con umbrales congelados; nada de esto vuelve a usarse para elegir.

In [4]:
TEST = {}; TD = {}
for m in MODELS:
    mod = YOLO(str(WTS[m])); fz = VAL[m]["estricta"]; za = VAL[m]["amplia"]
    v = mod.val(data=str(DSR / "data_base.yaml"), split="test", imgsz=320, conf=0.001, iou=0.7, max_det=1000, device="cpu", verbose=False, plots=False)
    per = {CLASS_NAMES[c]: dict(ap50=float(v.box.ap50[k]), p=float(v.box.p[k]), r=float(v.box.r[k])) for k, c in enumerate(v.box.ap_class_index)}
    dets = predict_dets(mod, IM["test"], 320, conf=0.01, iou=fz["iou"], device="cpu"); TD[m] = dets
    mb = match_boxes(dets, GT["test"], fz["conf"]); s = summarize(mb)
    amp = count_metrics(predict_dets(mod, IM["test"], 320, conf=0.01, iou=za["iou"], device="cpu"), GT["test"], za["conf"], [GB, GPB])
    TEST[m] = dict(seed=MODELS[m], thr=VAL[m], per_class=per, mAP50_todas=float(v.box.map50), estricta=s, amplia_mae=amp["mae"]); print(m, json.dumps(TEST[m], default=float)[:900])
agg = {k: dict(mean=float(np.mean([TEST[m]["estricta"][k] for m in MODELS])), sd=float(np.std([TEST[m]["estricta"][k] for m in MODELS], ddof=1))) for k in ("precision", "recall", "f1", "mae")}
agg["mAP50_Gb"] = dict(mean=float(np.mean([TEST[m]["per_class"]["G-bacilli"]["ap50"] for m in MODELS])), sd=float(np.std([TEST[m]["per_class"]["G-bacilli"]["ap50"] for m in MODELS], ddof=1)))
fz = VAL[CHOSEN]["estricta"]; dets = TD[CHOSEN]; mb = match_boxes(dets, GT["test"], fz["conf"])
detail = dict(modelo=CHOSEN, umbrales=VAL[CHOSEN], test=TEST[CHOSEN], ic95_grupos=boot_groups(mb), por_semilla_agregado=agg)
cm = confusion(dets, GT["test"], fz["conf"]); cm.to_csv(OUT / "confusion_test.csv"); print(cm.to_string())
for pat in ("crop", "simple"):
    sub = mb[mb.index.map(PATTERN) == pat]; detail[f"patron_{pat}"] = dict(**summarize(sub), ic95=boot_groups(sub))
# baselines en test
tr_med = float(INFO[INFO.split == "train"][["n_c2"]].median().iloc[0]); yt = np.array([int((GT["test"][k][0] == GB).sum()) for k in GT["test"]])
detail["baseline_mediana_test"] = dict(mediana_train=tr_med, mae=float(np.abs(yt - tr_med).mean()))
def lab_count(path, t=20, amin=300):
    a = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2LAB)[..., 1].astype(np.int16); da = cv2.GaussianBlur((a - int(np.median(a))).astype(np.float32), (0, 0), 1.5)
    mm = cv2.morphologyEx((da > t).astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)); n, _, st, _ = cv2.connectedComponentsWithStats(mm); return int((st[1:, cv2.CC_STAT_AREA] >= amin).sum())
detail["baseline_Lab_test_mae"] = float(np.abs(np.array([lab_count(p) for p in IM["test"]]) - yt).mean())
# fondos sinteticos de test: proxy sesgado (hay bacterias sin etiquetar en las ventanas)
bgi = sorted((DSR / "images" / "bg_test").glob("*.jpg")); dbg = predict_dets(YOLO(str(WTS[CHOSEN])), bgi, 320, conf=0.01, iou=fz["iou"], device="cpu")
nbg = np.array([int(((c == GB) & (s >= fz["conf"])).sum()) for c, s, _ in dbg.values()]); detail["fondos_bg_test_proxy"] = dict(n=len(nbg), fp_por_imagen=float(nbg.mean()), imgs_con_fp=float((nbg > 0).mean()), aviso="ventanas con bacterias sin etiquetar: cota superior")
(OUT / "test_final.json").write_text(json.dumps(detail, indent=1, default=float)); print(json.dumps(detail, indent=1, default=float))
mb.to_csv(OUT / "test_por_imagen_elegido.csv")

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2232.0±847.3 MB/s, size: 111.2 KB)
val: Scanning /kaggle/temp/yolo_ds/labels/test... 890 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 890/890 1.3Kit/s 0.7s0.1ss
val: New cache created: /kaggle/temp/yolo_ds/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.7it/s 20.5s0.4ss
                   all        890       1727      0.795      0.832      0.892      0.579
Speed: 0.3ms preprocess, 18.2ms inference, 0.0ms loss, 0.3ms postprocess per image
E3 {"seed": 42, "thr": {"estricta": {"iou": 0.5, "conf": 0.6, "mae_val": 0.29591836734693877}, "amplia": {"iou": 0.5, "conf": 0.5, "mae_val": 0.35034013605442177}, "val_estricta": {"precision": 0.8502358490566038, "recall": 0.7931793179317932, "f1

## Paso 8a. Exportacion del elegido: ONNX, TFLite FP32, FP16 (float_casting) e INT8 (calibracion 300 imagenes de train)

In [5]:
EX = Path("/kaggle/working/export"); EX.mkdir(exist_ok=True); W = EX / f"ecoli_gb_{CHOSEN}.pt"; shutil.copy2(WTS[CHOSEN], W)
tr = INFO[INFO.split == "train"].copy(); rng = np.random.default_rng(42)
tr["qg"] = pd.qcut(tr.gray.rank(method="first"), 4, labels=False); tr["qb"] = pd.qcut(tr.lab_b.rank(method="first"), 4, labels=False)
cal = pd.concat([g.sample(min(len(g), 19), random_state=42) for _, g in tr.groupby(["qg", "qb"])]).head(300)
if len(cal) < 300: cal = pd.concat([cal, tr.drop(cal.index).sample(300 - len(cal), random_state=42)])
(EX / "calib.txt").write_text("\n".join(str(DSR / "images" / "train" / n) for n in cal.name)); print("calibracion:", len(cal), "imagenes", cal.groupby(["qg", "qb"]).size().describe()[["min", "max"]].to_dict())
(EX / "calib.yaml").write_text(f"path: {DSR}\ntrain: {EX / 'calib.txt'}\nval: {EX / 'calib.txt'}\nnames:\n{names}\n")
def run_export(tag, kw, timeout=1500):
    code = f"from ultralytics import YOLO; print('EXPORT OK', YOLO('{W}').export(imgsz=320, {kw}))"; t = time.time()
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=timeout, cwd=str(EX)); print(f"[{tag}] rc={r.returncode} {time.time()-t:.0f}s", (r.stdout + r.stderr)[-500:], flush=True); return r.returncode == 0
    except subprocess.TimeoutExpired: print(f"[{tag}] TIMEOUT", flush=True); return False
ok = {"onnx": run_export("onnx", "format='onnx', simplify=True"), "fp32": run_export("tflite fp32", "format='tflite'"), "int8": run_export("tflite int8", f"format='tflite', int8=True, data='{EX / 'calib.yaml'}'")}
print(sorted(p.name for p in EX.glob("*")))
fp32 = EX / f"ecoli_gb_{CHOSEN}.tflite"
if fp32.exists(): shutil.copy2(fp32, EX / f"ecoli_gb_{CHOSEN}_fp32.tflite")
try:
    from ai_edge_quantizer import quantizer, qtyping
    q = quantizer.Quantizer(str(EX / f"ecoli_gb_{CHOSEN}_fp32.tflite"))
    cfg = qtyping.OpQuantizationConfig(weight_tensor_config=qtyping.TensorQuantizationConfig(num_bits=16, dtype=qtyping.TensorDataType.FLOAT), compute_precision=qtyping.ComputePrecision.FLOAT, explicit_dequantize=True)
    q.update_quantization_recipe(regex=".*", operation_name=qtyping.TFLOperationName.ALL_SUPPORTED, algorithm_key="float_casting", op_config=cfg)
    q.quantize().export_model(str(EX / f"ecoli_gb_{CHOSEN}_fp16.tflite"), overwrite=True); ok["fp16"] = True
except Exception as e: print("FP16 fallo", repr(e)[:400]); ok["fp16"] = False
print(ok)

calibracion: 300 imagenes {'min': 15.0, 'max': 19.0}
[onnx] rc=0 9s ccess ✅ 3.3s, saved as '/kaggle/working/export/ecoli_gb_E3s43.onnx' (12.1 MB)

Export complete (3.6s)
Results saved to /kaggle/working/export/ecoli_gb_E3s43.onnx
Predict:         yolo predict task=detect model=/kaggle/working/export/ecoli_gb_E3s43.onnx imgsz=320 
Validate:        yolo val task=detect model=/kaggle/working/export/ecoli_gb_E3s43.onnx imgsz=320 data=/kaggle/temp/yolo_ds/data_base.yaml 
Visualize:       https://netron.app
EXPORT OK /kaggle/working/export/ecoli_gb_E3s43.onnx

[tflite fp32] rc=0 25s e `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(tr

Applying Transformations to tensors:: 100%|██████████| 415/415 [00:00<00:00, 157423.91it/s]

Model name: /kaggle/working/export/ecoli_gb_E3s43_fp32.tflite
Original model size: 12.10 MiB
Quantized model size: 6.15 MiB
Quantization Ratio: 0.51 (2.0x smaller)
Total time: 87.31 ms
{'onnx': True, 'fp32': True, 'int8': True, 'fp16': True}


## Paso 8b. Paridad en 300 imagenes de test (pt, ONNX, FP32, FP16, INT8): mAP50 G-bacilli y conteo; I/O de los TFLite; latencia proxy

In [6]:
rng = np.random.default_rng(42); sub = [IM["test"][i] for i in sorted(rng.choice(len(IM["test"]), 300, replace=False))]
(EX / "par300.txt").write_text("\n".join(map(str, sub))); (EX / "par300.yaml").write_text(f"path: {DSR}\ntrain: {EX / 'par300.txt'}\nval: {EX / 'par300.txt'}\ntest: {EX / 'par300.txt'}\nnames:\n{names}\n")
G300 = {p.stem: GT["test"][p.stem] for p in sub}; zc = VAL[CHOSEN]["estricta"]
FILES = {"pt": W, "onnx": EX / f"ecoli_gb_{CHOSEN}.onnx", "tflite_fp32": EX / f"ecoli_gb_{CHOSEN}_fp32.tflite", "tflite_fp16": EX / f"ecoli_gb_{CHOSEN}_fp16.tflite"}
i8 = sorted(p for p in EX.glob("*int8*.tflite")); FILES["tflite_int8"] = i8[0] if i8 else EX / "missing_int8.tflite"
par = {}
for tag, f in FILES.items():
    if not Path(f).exists(): par[tag] = dict(error="no existe"); continue
    try:
        mod = YOLO(str(f), task="detect"); v = mod.val(data=str(EX / "par300.yaml"), split="val", imgsz=320, conf=0.001, iou=0.7, max_det=1000, device="cpu", verbose=False, plots=False)
        k = list(v.box.ap_class_index).index(GB); d = predict_dets(mod, sub, 320, conf=0.01, iou=zc["iou"], device="cpu"); s = summarize(match_boxes(d, G300, zc["conf"]))
        t = []; im0 = str(sub[0])
        for _ in range(5): mod.predict(im0, imgsz=320, device="cpu", verbose=False)
        for _ in range(50):
            t0 = time.perf_counter(); mod.predict(im0, imgsz=320, device="cpu", verbose=False); t.append((time.perf_counter() - t0) * 1000)
        par[tag] = dict(mAP50_Gb=float(v.box.ap50[k]), **{x: s[x] for x in ("precision", "recall", "f1", "mae")}, lat_ms_mediana_e2e=float(np.median(t)), mb=round(Path(f).stat().st_size / 1e6, 2))
    except Exception as e: par[tag] = dict(error=repr(e)[:300])
    print(tag, par[tag], flush=True)
(OUT / "paridad_300_test.json").write_text(json.dumps(par, indent=1))
io = {}
try:
    import tensorflow as tf
    for tag in ("tflite_fp32", "tflite_fp16", "tflite_int8"):
        if not Path(FILES[tag]).exists(): continue
        it = tf.lite.Interpreter(model_path=str(FILES[tag])); it.allocate_tensors()
        io[tag] = dict(inputs=[dict(shape=d["shape"].tolist(), dtype=d["dtype"].__name__, quant=[float(d["quantization"][0]), int(d["quantization"][1])]) for d in it.get_input_details()],
                       outputs=[dict(shape=d["shape"].tolist(), dtype=d["dtype"].__name__, quant=[float(d["quantization"][0]), int(d["quantization"][1])]) for d in it.get_output_details()])
except Exception as e: io["error"] = repr(e)[:300]
print(json.dumps(io, indent=1)); (OUT / "tflite_io.json").write_text(json.dumps(io, indent=1))

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2520.6±430.9 MB/s, size: 102.4 KB)
val: Scanning /kaggle/temp/yolo_ds/labels/test... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 1.3Kit/s 0.2s<0.1s
val: New cache created: /kaggle/temp/yolo_ds/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.7it/s 6.9s0.4s
                   all        300        586      0.803      0.802       0.87       0.56
Speed: 0.3ms preprocess, 18.3ms inference, 0.0ms loss, 0.3ms postprocess per image
pt {'mAP50_Gb': 0.9525265657074645, 'precision': 0.8524590163934426, 'recall': 0.8062015503875969, 'f1': 0.8286852589641434, 'mae': 0.24, 'lat_ms_mediana_e2e': 30.024489000027188, 'mb': 6.47}
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu128 C

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 300/300 34.6it/s 8.7s0.1s
                   all        300        586      0.761      0.783       0.84      0.537
Speed: 0.2ms preprocess, 23.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Loading /kaggle/working/export/ecoli_gb_E3s43_fp32.tflite for LiteRT inference...
tflite_fp32 {'mAP50_Gb': 0.9058344084730917, 'precision': 0.8524590163934426, 'recall': 0.8062015503875969, 'f1': 0.8286852589641434, 'mae': 0.24, 'lat_ms_mediana_e2e': 28.683474500098782, 'mb': 12.69}
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Loading /kaggle/working/export/ecoli_gb_E3s43_fp16.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2206.6±382.3 MB/s, size: 96.1 KB)
val: Scanning /kaggle/temp/yolo_ds/labels/test.cache... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


929

## Artefactos y sha256

In [7]:
for f in EX.glob("*"):
    if f.is_file() and f.suffix in (".pt", ".onnx", ".tflite", ".txt", ".yaml"): shutil.copy2(f, OUT / f.name)
sha = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob("ecoli_gb_*") if p.suffix in (".pt", ".onnx", ".tflite")}
(OUT / "sha256_pesos.json").write_text(json.dumps(sha, indent=1)); print(json.dumps(sha, indent=1))
print(subprocess.run("pip freeze | grep -i -E '^(ultralytics|torch|tensorflow|onnx|onnx2tf|onnxruntime|numpy|ai-edge)'", shell=True, capture_output=True, text=True).stdout)
(OUT / "pip_freeze_final.txt").write_text(subprocess.run("pip freeze | grep -i -E '^(ultralytics|torch|tensorflow|onnx|onnx2tf|onnxruntime|numpy|ai-edge|opencv|scikit|scipy|pandas)'", shell=True, capture_output=True, text=True).stdout)

{
 "ecoli_gb_E3s43_fp16.tflite": "28fcc93a479aadd92e038d45d8bde4e9b20cf9ad9e0b07735cef4c832fca2621",
 "ecoli_gb_E3s43_int8.tflite": "b1c2a33537e841fa2637d6fe6a5b23c09b9f6fad8c6c3fdab5d3867f8aa80b28",
 "ecoli_gb_E3s43_fp32.tflite": "5ca4a1a3392cb9c54d8cca0a09c23ac2bbf91d531911e5cb906f36ce8d4eb383",
 "ecoli_gb_E3s43.tflite": "5ca4a1a3392cb9c54d8cca0a09c23ac2bbf91d531911e5cb906f36ce8d4eb383",
 "ecoli_gb_E3s43.onnx": "cf485e19e4035c800440332c9907fccac533716fce5de0dfed568e12f49a63a1",
 "ecoli_gb_E3s43.pt": "c1f9732de4b86db00e21024620743d94512a8842aa80d5b00622d99d43c72730"
}
ai-edge-litert==2.2.0
ai-edge-quantizer==0.9.0
numpy==2.1.3
onnx==1.22.0
onnxruntime==1.31.0
onnxslim==0.1.98
tensorflow==2.20.0
tensorflow-datasets==4.9.10
tensorflow-hub==0.16.1
tensorflow-metadata==1.21.0
tensorflow-probability==0.25.0
tensorflow-text==2.20.1
torch==2.11.0+cu128
torchao==0.18.0
torchaudio==2.11.0+cu128
torchcodec==0.11.0+cu128
torchdata==0.11.0
torchinfo==1.8.0
torchmetrics==1.9.0
torchsummary==1.5.1


905